## Q1. Embedding a query:
Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value
(`v[0]`)?

* -0.31
* -0.02
* 0.12
* 0.44

In [72]:
from onnx_llm.embedder import Embedder
embed = Embedder("onnx_llm/models/Xenova/all-MiniLM-L6-v2")

q1 = "How does approximate nearest neighbor search work?"
v = embed.encode(q1)
v[0]

np.float64(-0.02058203437252893)

In [73]:
v.shape

(384,)

## Loading the data:

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [11]:
documents[0].keys()

dict_keys(['content', 'filename'])

## Cosine Similarity:

The embedder returns normalized vectors, so the dot product between two
of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed
its `content`, and compute the cosine similarity with the query vector
from Q1. What do you get?

* 0.07
* 0.37
* 0.68
* 0.92

In [30]:
sql_search = [doc for doc in documents if doc['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md']
content = embed.encode(sql_search[0]['content'])
v.dot(content)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand
Which file does the highest-scoring chunk belong to (its filename)?

* 02-vector-search/lessons/03-embeddings-dataset.md
* 02-vector-search/lessons/06-rag-vector.md
* 02-vector-search/lessons/07-sqlitesearch-vector.md
* 02-vector-search/lessons/09-onnx-embedder.md

In [67]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunks[0].keys()

dict_keys(['start', 'content', 'filename'])

In [68]:
texts = [doc["content"] for doc in chunks]
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
X.shape

  0%|          | 0/6 [00:00<?, ?it/s]

(295, 384)

In [74]:
scores = X.dot(v)
idx = np.argmax(scores)
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch
Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

* 02-vector-search/lessons/04-vector-search.md
* 04-evaluation/lessons/05-search-metrics.md
* 04-evaluation/lessons/13-llm-as-judge.md
* 05-monitoring/lessons/04-metrics.md

In [43]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields = ['content'])
vindex.fit(X, documents) 

In [80]:
q4_query = 'What metric do we use to evaluate a search engine?'
q4_vector = embed.encode(q4_query)

In [81]:
results = vindex.search(q4_vector, num_results=1)
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search
Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

* 02-vector-search/lessons/01-intro.md
* 02-vector-search/lessons/02-embeddings.md
* 02-vector-search/lessons/08-pgvector.md
* 03-orchestration/lessons/05-rag.md

In [61]:
import os
import sys
sys.path.append("../Week_01_Agentic_RAG")
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
import os 
load_dotenv()
from minsearch import Index

In [82]:
q5_query = 'How do I store vectors in PostgreSQL?'
q5_vector = embed.encode(q5_query)

In [79]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"])
text_index.fit(chunks)
text_filename = [t['filename'] for t in text_index.search(q5_query, num_results=5)]
text_filename

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [84]:
vector_filename = [v['filename'] for v in vindex.search(q5_vector, num_results=5)]
vector_filename

['02-vector-search/lessons/08-pgvector.md',
 '05-monitoring/lessons/05-database.md',
 '02-vector-search/lessons/02-embeddings.md',
 '02-vector-search/lessons/04-vector-search.md',
 '02-vector-search/lessons/07-sqlitesearch-vector.md']

In [85]:
[i for i in vector_filename if i not in text_filename]

['02-vector-search/lessons/08-pgvector.md',
 '05-monitoring/lessons/05-database.md',
 '02-vector-search/lessons/04-vector-search.md',
 '02-vector-search/lessons/07-sqlitesearch-vector.md']

## Q6. Hybrid search
Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))

In [86]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [88]:
q6_query = "How do I give the model access to tools?"
q6_vector = embed.encode(q6_query)

In [104]:
vindex.fit(X, chunks)

In [105]:
text_results = text_index.search(q6_query, num_results=5)
vector_results = vindex.search(q6_vector, num_results=5)

In [107]:
print("TEXT")

for i, doc in enumerate(text_results):
    print(i, doc["filename"], doc["start"])

print()

print("VECTOR")

for i, doc in enumerate(vector_results):
    print(i, doc["filename"], doc["start"])

TEXT
0 01-agentic-rag/lessons/14-agentic-loop.md 0
1 01-agentic-rag/lessons/13-function-calling.md 4000
2 01-agentic-rag/lessons/13-function-calling.md 5000
3 01-agentic-rag/lessons/13-function-calling.md 1000
4 04-evaluation/lessons/02-ground-truth.md 3000

VECTOR
0 01-agentic-rag/lessons/01-intro.md 2000
1 04-evaluation/lessons/02-ground-truth.md 1000
2 01-agentic-rag/lessons/16-other-frameworks.md 0
3 01-agentic-rag/lessons/15-frameworks.md 2000
4 01-agentic-rag/lessons/13-function-calling.md 4000


In [110]:
rrf([vector_results, text_results])[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'